In [1]:
import os
import math
import random
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
SEED = 42

BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5

LOOKBACK = 96

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cuda


In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

print("Seed:", SEED)

Seed: 42


In [4]:
DATA_DIR = "/kaggle/input/datasets/vedantjadhavv/electricity-data"

DATASETS = {
    "ETTh1": os.path.join(DATA_DIR, "ETTh1.csv"),
    "ETTh2": os.path.join(DATA_DIR, "ETTh2.csv"),
    "ETTm1": os.path.join(DATA_DIR, "ETTm1.csv"),
    "ETTm2": os.path.join(DATA_DIR, "ETTm2.csv"),
    "Electricity": os.path.join(DATA_DIR, "electricity.csv"),
}

for name, path in DATASETS.items():
    print(name, "->", path, "| exists:", os.path.exists(path))

ETTh1 -> /kaggle/input/datasets/vedantjadhavv/electricity-data/ETTh1.csv | exists: True
ETTh2 -> /kaggle/input/datasets/vedantjadhavv/electricity-data/ETTh2.csv | exists: True
ETTm1 -> /kaggle/input/datasets/vedantjadhavv/electricity-data/ETTm1.csv | exists: True
ETTm2 -> /kaggle/input/datasets/vedantjadhavv/electricity-data/ETTm2.csv | exists: True
Electricity -> /kaggle/input/datasets/vedantjadhavv/electricity-data/electricity.csv | exists: True


In [5]:
for name, path in DATASETS.items():

    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist()[:10])

    print("\nFirst rows:")
    display(df.head(3))


ETTh1
Shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

First rows:


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001



ETTh2
Shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

First rows:


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,41.130001,12.481,36.535999,9.355,4.424,1.311,38.661999
1,2016-07-01 01:00:00,37.528000,10.136,33.936001,7.532,4.435,1.215,37.124001
2,2016-07-01 02:00:00,37.946999,11.309,35.330002,9.007,2.100,0.000,36.465000



ETTm1
Shape: (69680, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

First rows:


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 00:15:00,5.760,2.076,1.492,0.426,4.264,1.401,30.459999
2,2016-07-01 00:30:00,5.760,1.942,1.492,0.391,4.234,1.310,30.038000



ETTm2
Shape: (69680, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

First rows:


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,41.130001,12.481,36.535999,9.355,4.424,1.311,38.661999
1,2016-07-01 00:15:00,39.622002,11.309,35.543999,8.551,3.209,1.258,38.223000
2,2016-07-01 00:30:00,38.868000,10.555,34.365002,7.586,4.435,1.258,37.344002



Electricity
Shape: (26304, 321)
Columns: ['date', 'MT_001', 'MT_002', 'MT_003', 'MT_004', 'MT_005', 'MT_006', 'MT_007', 'MT_008', 'MT_009']

First rows:


,date,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,...,MT_360,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369
0,2012-01-01 00:00:00,3.172589,17.069701,57.993050,103.150407,53.353659,263.392857,6.783493,209.595960,55.944056,...,168.859649,92.612420,20025.0,1179.324895,1250.000000,11.734029,9.215916,389.157155,45.075125,540.139296
1,2012-01-01 01:00:00,4.124365,22.581792,77.324066,138.211382,72.256098,340.029762,6.783493,275.252525,67.307692,...,200.932018,112.598144,23800.0,1160.337553,1653.409091,15.971317,11.263897,543.898156,62.604341,708.211144
2,2012-01-01 02:00:00,4.758883,23.470839,77.324066,139.227642,67.378049,309.523810,6.924816,255.892256,66.870629,...,203.947368,107.066381,24150.0,1070.675105,1642.045455,15.645372,10.239906,547.848990,53.839733,690.249267


In [6]:
class TimeSeriesWindowDataset(Dataset):

    def __init__(self, data, lookback, horizon):
        self.data = torch.tensor(data, dtype=torch.float32)

        self.lookback = lookback
        self.horizon = horizon

        self.length = len(data) - lookback - horizon + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):

        x = self.data[
            idx : idx + self.lookback
        ]

        y = self.data[
            idx + self.lookback :
            idx + self.lookback + self.horizon
        ]

        return x, y

In [7]:
def load_dataframe(path):

    df = pd.read_csv(path)

    # First column is timestamp for ETT/Electricity
    first_col = df.columns[0]

    try:
        pd.to_datetime(df[first_col])
        timestamp_col = first_col
    except:
        timestamp_col = None

    if timestamp_col is not None:
        df = df.drop(columns=[timestamp_col])

    # Keep numerical columns only
    df = df.select_dtypes(include=[np.number])

    # Remove columns that contain no useful values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(axis=1, how="all")

    # Fill remaining missing values
    df = df.interpolate(limit_direction="both")

    return df


def make_dataloaders(
    path,
    lookback,
    horizon,
    batch_size=64,
    train_ratio=0.7,
    val_ratio=0.1,
    feature_limit=None
):

    df = load_dataframe(path)

    # Optional feature restriction.
    # None = use all available numerical features.
    if feature_limit is not None:
        df = df.iloc[:, :feature_limit]

    values = df.values.astype(np.float32)

    n = len(values)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_raw = values[:train_end]
    val_raw = values[train_end:val_end]
    test_raw = values[val_end:]

    # Normalize using TRAIN ONLY
    mean = train_raw.mean(axis=0)
    std = train_raw.std(axis=0)

    std[std < 1e-6] = 1.0

    train_data = (train_raw - mean) / std
    val_data = (val_raw - mean) / std
    test_data = (test_raw - mean) / std

    train_dataset = TimeSeriesWindowDataset(
        train_data,
        lookback,
        horizon
    )

    val_dataset = TimeSeriesWindowDataset(
        val_data,
        lookback,
        horizon
    )

    test_dataset = TimeSeriesWindowDataset(
        test_data,
        lookback,
        horizon
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=2,
        pin_memory=True
    )

    stats = {
        "mean": mean,
        "std": std,
        "num_features": values.shape[1],
        "total_rows": n,
    }

    return (
        train_loader,
        val_loader,
        test_loader,
        stats
    )

In [8]:
train_loader, val_loader, test_loader, stats = make_dataloaders(
    DATASETS["ETTh1"],
    lookback=96,
    horizon=96,
    batch_size=BATCH_SIZE
)

print("Features:", stats["num_features"])
print("Rows:", stats["total_rows"])

x, y = next(iter(train_loader))

print("X shape:", x.shape)
print("Y shape:", y.shape)

Features: 7
Rows: 17420
X shape: torch.Size([64, 96, 7])
Y shape: torch.Size([64, 96, 7])


In [27]:
ETT_DATASETS = [
    "ETTh1",
    "ETTh2",
    "ETTm1",
    "ETTm2"
]

HORIZONS = [
    96,
    192,
    336,
    720
]

MAIN_MODELS = [
    "Linear",
    "NLinear",
    "DLinear",
    "PatchMLPTS",
    "PatchTST"
]

results = []


def create_model(
    name,
    lookback,
    horizon,
    num_features
):

    if name == "Linear":
        return LinearModel(
            lookback,
            horizon,
            num_features
        )

    if name == "NLinear":
        return NLinearModel(
            lookback,
            horizon,
            num_features
        )

    if name == "DLinear":
        return DLinearModel(
            lookback,
            horizon,
            num_features,
            moving_avg_kernel=25
        )

    if name == "PatchMLPTS":
        return PatchMLPTS(
            lookback,
            horizon,
            num_features,
            patch_len=12,
            embed_dim=128,
            num_blocks=3,
            dropout=0.1
        )

    if name == "PatchTST":
        return PatchTST(
            lookback,
            horizon,
            num_features,
            patch_len=16,
            stride=8,
            d_model=128,
            n_heads=8,
            num_layers=2,
            dropout=0.1
        )

    raise ValueError(name)

In [28]:
for dataset_name in ETT_DATASETS:

    path = DATASETS[dataset_name]

    for horizon in HORIZONS:

        print("\n" + "=" * 70)
        print(
            f"{dataset_name} | horizon={horizon}"
        )
        print("=" * 70)

        train_loader, val_loader, test_loader, stats = make_dataloaders(
            path,
            lookback=LOOKBACK,
            horizon=horizon,
            batch_size=BATCH_SIZE
        )

        num_features = stats["num_features"]

        for model_name in MAIN_MODELS:

            print(
                f"\n--- {model_name} ---"
            )

            seed_everything(SEED)

            model = create_model(
                model_name,
                LOOKBACK,
                horizon,
                num_features
            )

            params = count_parameters(model)

            model, test_mse, test_mae, history = fit_model(
                model,
                train_loader,
                val_loader,
                test_loader,
                epochs=EPOCHS,
                lr=LR,
                weight_decay=WEIGHT_DECAY,
                patience=PATIENCE,
                device=DEVICE,
                verbose=False
            )

            print(
                f"{model_name}: "
                f"MSE={test_mse:.4f} "
                f"MAE={test_mae:.4f} "
                f"params={params:,}"
            )

            results.append({
                "dataset": dataset_name,
                "lookback": LOOKBACK,
                "horizon": horizon,
                "model": model_name,
                "parameters": params,
                "MSE": test_mse,
                "MAE": test_mae
            })

            del model
            torch.cuda.empty_cache()


df_ett_results = pd.DataFrame(results)

df_ett_results.to_csv(
    "ett_benchmark_results.csv",
    index=False
)

print("\nSaved:")
print("ett_benchmark_results.csv")


ETTh1 | horizon=96

--- Linear ---
Linear: MSE=0.4415 MAE=0.4457 params=9,312

--- NLinear ---
NLinear: MSE=0.4462 MAE=0.4492 params=9,312

--- DLinear ---
DLinear: MSE=0.4374 MAE=0.4427 params=18,624

--- PatchMLPTS ---
PatchMLPTS: MSE=0.4459 MAE=0.4614 params=100,752

--- PatchTST ---
PatchTST: MSE=0.4249 MAE=0.4484 params=534,240

ETTh1 | horizon=192

--- Linear ---
Linear: MSE=0.4928 MAE=0.4847 params=18,624

--- NLinear ---
NLinear: MSE=0.5059 MAE=0.4896 params=18,624

--- DLinear ---
DLinear: MSE=0.4896 MAE=0.4823 params=37,248

--- PatchMLPTS ---
PatchMLPTS: MSE=0.4819 MAE=0.4855 params=199,152

--- PatchTST ---
PatchTST: MSE=0.4730 MAE=0.4870 params=669,504

ETTh1 | horizon=336

--- Linear ---
Linear: MSE=0.5365 MAE=0.5188 params=32,592

--- NLinear ---
NLinear: MSE=0.5503 MAE=0.5178 params=32,592

--- DLinear ---
DLinear: MSE=0.5349 MAE=0.5175 params=65,184

--- PatchMLPTS ---
PatchMLPTS: MSE=0.5388 MAE=0.5242 params=346,752

--- PatchTST ---
PatchTST: MSE=0.5081 MAE=0.5148 p

In [29]:
df_ett_results

,dataset,lookback,horizon,model,parameters,MSE,MAE
0,ETTh1,96,96,Linear,9312,0.441476,0.445740
1,ETTh1,96,96,NLinear,9312,0.446228,0.449209
2,ETTh1,96,96,DLinear,18624,0.437396,0.442730
3,ETTh1,96,96,PatchMLPTS,100752,0.445950,0.461434
4,ETTh1,96,96,PatchTST,534240,0.424915,0.448354
...,...,...,...,...,...,...,...
75,ETTm2,96,720,Linear,69840,0.227377,0.328480
76,ETTm2,96,720,NLinear,69840,0.238323,0.328228
77,ETTm2,96,720,DLinear,139680,0.226193,0.326915
78,ETTm2,96,720,PatchMLPTS,740352,0.227926,0.329658


In [30]:
winner_mse = (
    df_ett_results
    .loc[
        df_ett_results
        .groupby(
            ["dataset", "horizon"]
        )["MSE"]
        .idxmin()
    ]
    .sort_values(
        ["dataset", "horizon"]
    )
)

print("MSE WINNERS")

display(
    winner_mse[
        [
            "dataset",
            "horizon",
            "model",
            "MSE",
            "MAE"
        ]
    ]
)

MSE WINNERS


,dataset,horizon,model,MSE,MAE
4,ETTh1,96,PatchTST,0.424915,0.448354
9,ETTh1,192,PatchTST,0.473046,0.487046
14,ETTh1,336,PatchTST,0.508094,0.514848
19,ETTh1,720,PatchTST,0.649954,0.604900
20,ETTh2,96,Linear,0.168431,0.281998
27,ETTh2,192,DLinear,0.193909,0.305255
32,ETTh2,336,DLinear,0.212184,0.327748
38,ETTh2,720,PatchMLPTS,0.259562,0.366039
42,ETTm1,96,DLinear,0.354617,0.384189
45,ETTm1,192,Linear,0.422984,0.420580


In [31]:
ABLATION_DATASETS = [
    "ETTh1",
    "ETTh2"
]

ABLATION_MODELS = [
    "PatchMLPTS",
    "PatchMLPTS_NoPatching",
    "PatchMLPTS_ChannelMixing"
]

ablation_results = []

In [32]:
for dataset_name in ABLATION_DATASETS:

    path = DATASETS[dataset_name]

    for horizon in HORIZONS:

        print("\n" + "=" * 70)
        print(
            f"ABLATION | {dataset_name} | H={horizon}"
        )
        print("=" * 70)

        train_loader, val_loader, test_loader, stats = make_dataloaders(
            path,
            lookback=LOOKBACK,
            horizon=horizon,
            batch_size=BATCH_SIZE
        )

        C = stats["num_features"]

        for model_name in ABLATION_MODELS:

            print(
                f"\n--- {model_name} ---"
            )

            seed_everything(SEED)

            if model_name == "PatchMLPTS":

                model = PatchMLPTS(
                    LOOKBACK,
                    horizon,
                    C,
                    patch_len=12,
                    embed_dim=128,
                    num_blocks=3,
                    dropout=0.1
                )

            elif model_name == "PatchMLPTS_NoPatching":

                model = PatchMLPTS_NoPatching(
                    LOOKBACK,
                    horizon,
                    C,
                    embed_dim=128,
                    num_blocks=3,
                    dropout=0.1
                )

            elif model_name == "PatchMLPTS_ChannelMixing":

                model = PatchMLPTS_ChannelMixing(
                    LOOKBACK,
                    horizon,
                    C,
                    patch_len=12,
                    embed_dim=128,
                    num_blocks=3,
                    dropout=0.1
                )

            params = count_parameters(model)

            model, test_mse, test_mae, history = fit_model(
                model,
                train_loader,
                val_loader,
                test_loader,
                epochs=EPOCHS,
                lr=LR,
                weight_decay=WEIGHT_DECAY,
                patience=PATIENCE,
                device=DEVICE,
                verbose=False
            )

            print(
                f"MSE={test_mse:.4f} "
                f"MAE={test_mae:.4f}"
            )

            ablation_results.append({
                "dataset": dataset_name,
                "lookback": LOOKBACK,
                "horizon": horizon,
                "model": model_name,
                "parameters": params,
                "MSE": test_mse,
                "MAE": test_mae
            })

            del model
            torch.cuda.empty_cache()


df_ablation = pd.DataFrame(
    ablation_results
)

df_ablation.to_csv(
    "patchmlpts_ablation_results.csv",
    index=False
)

print(
    "\nSaved patchmlpts_ablation_results.csv"
)


ABLATION | ETTh1 | H=96

--- PatchMLPTS ---
MSE=0.4459 MAE=0.4614

--- PatchMLPTS_NoPatching ---
MSE=0.4232 MAE=0.4465

--- PatchMLPTS_ChannelMixing ---
MSE=0.4615 MAE=0.4736

ABLATION | ETTh1 | H=192

--- PatchMLPTS ---
MSE=0.4819 MAE=0.4855

--- PatchMLPTS_NoPatching ---
MSE=0.4690 MAE=0.4803

--- PatchMLPTS_ChannelMixing ---
MSE=0.5262 MAE=0.5179

ABLATION | ETTh1 | H=336

--- PatchMLPTS ---
MSE=0.5388 MAE=0.5242

--- PatchMLPTS_NoPatching ---
MSE=0.5267 MAE=0.5220

--- PatchMLPTS_ChannelMixing ---
MSE=0.5678 MAE=0.5466

ABLATION | ETTh1 | H=720

--- PatchMLPTS ---
MSE=0.6671 MAE=0.6061

--- PatchMLPTS_NoPatching ---
MSE=0.6369 MAE=0.5934

--- PatchMLPTS_ChannelMixing ---
MSE=0.7186 MAE=0.6399

ABLATION | ETTh2 | H=96

--- PatchMLPTS ---
MSE=0.1723 MAE=0.2831

--- PatchMLPTS_NoPatching ---
MSE=0.1784 MAE=0.2912

--- PatchMLPTS_ChannelMixing ---
MSE=0.1830 MAE=0.3139

ABLATION | ETTh2 | H=192

--- PatchMLPTS ---
MSE=0.1985 MAE=0.3118

--- PatchMLPTS_NoPatching ---
MSE=0.2047 MAE=0.3

In [33]:
ablation_pivot = df_ablation.pivot_table(
    index=["dataset", "horizon"],
    columns="model",
    values=["MSE", "MAE"]
)

ablation_pivot

MAE                                                 \
model           PatchMLPTS PatchMLPTS_ChannelMixing PatchMLPTS_NoPatching   
dataset horizon                                                             
ETTh1   96        0.461434                 0.473616              0.446546   
        192       0.485544                 0.517897              0.480255   
        336       0.524172                 0.546618              0.522040   
        720       0.606070                 0.639906              0.593380   
ETTh2   96        0.283078                 0.313878              0.291169   
        192       0.311760                 0.354462              0.311032   
        336       0.334205                 0.363149              0.324515   
        720       0.366039                 0.398390              0.344361   

                       MSE                                                 
model           PatchMLPTS PatchMLPTS_ChannelMixing PatchMLPTS_NoPatching  
dataset horizon                                                            
ETTh1   96        0.445950                 0.461502              0.423163  
        192       0.481887                 0.526171              0.468954  
        336       0.538798                 0.567760              0.526735  
        720       0.667077                 0.718607              0.636874  
ETTh2   96        0.172301                 0.182951              0.178441  
        192       0.198504                 0.225127              0.204746  
        336       0.217066                 0.235341              0.210492  
        720       0.259562                 0.278826              0.241442

In [34]:
electricity_path = DATASETS["Electricity"]

print("Electricity path:")
print(electricity_path)

electricity = pd.read_csv(
    electricity_path
)

print("\nShape:")
print(electricity.shape)

display(electricity.head())

Electricity path:
/kaggle/input/datasets/vedantjadhavv/electricity-data/electricity.csv

Shape:
(26304, 321)


,date,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,...,MT_360,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369
0,2012-01-01 00:00:00,3.172589,17.069701,57.993050,103.150407,53.353659,263.392857,6.783493,209.595960,55.944056,...,168.859649,92.612420,20025.0,1179.324895,1250.000000,11.734029,9.215916,389.157155,45.075125,540.139296
1,2012-01-01 01:00:00,4.124365,22.581792,77.324066,138.211382,72.256098,340.029762,6.783493,275.252525,67.307692,...,200.932018,112.598144,23800.0,1160.337553,1653.409091,15.971317,11.263897,543.898156,62.604341,708.211144
2,2012-01-01 02:00:00,4.758883,23.470839,77.324066,139.227642,67.378049,309.523810,6.924816,255.892256,66.870629,...,203.947368,107.066381,24150.0,1070.675105,1642.045455,15.645372,10.239906,547.848990,53.839733,690.249267
3,2012-01-01 03:00:00,4.441624,22.226174,77.324066,110.264228,52.743902,210.565476,5.652911,207.912458,44.143357,...,199.835526,72.626695,23625.0,1054.852321,1590.909091,15.971317,9.215916,328.358209,47.996661,683.284457
4,2012-01-01 04:00:00,4.758883,21.870555,77.324066,85.873984,47.256098,161.458333,3.533070,182.659933,46.328671,...,201.480263,69.236260,22825.0,1028.481013,1573.863636,18.252934,9.508484,343.942054,47.161937,679.985337
